# Eflomal, End-to-End

This bundles up the whole process from end-to-end for reuse. It does not describe the details: for that, see [Automated Alignments](30AutoAligners.ipynb) for scoring the results. 

Assumptions:
* Piped files already exist in `autoalignment/data/<LANG>/<TARGETID>`, both `<SOURCEID>-<TARGETID>.piped.txt` and `<SOURCEID>-<TARGETID>-lemma.piped.txt`
* Running `eflomal` currently only works in `internal-alignments`: we need to port it over to `biblealignlib`. 

Overview:
* Get latest data from ClearAligner
* Run automated alignment
* Score automated alignment
* Visualize automated alignment


## Get Data from ClearAligner

* Sync project
* Edit > Export Project
* Rename to `<SOURCEID>-<TARGETID>-<DATE>manual.json`
* Move to `alignments-<LANG>/data/alignments/<TARGETID>`

In [1]:
# PARAMETERS: your local copy of alignments-eng/data
(targetlang, targetid, sourceid) = ("ben", "IRVBen", "SBLGNT")
# best practice: include algorithm and date
condition = "20250110eflomal"


from biblealignlib.burrito import CLEARROOT, AlignmentSet
from biblealignlib.autoalign import scorer, reader
# bundled parameters for downstream processes
alset = AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(CLEARROOT / f"alignments-{targetlang}/data"))


## Running Automated Alignment

Details: [30AutoAligners > Running Automated Alignment](http://localhost:8888/lab/workspaces/auto-U/tree/notebooks/30AutoAligners.ipynb#Running-Automated-Alignment)

In [6]:
# this will create the directory if it doesn't already exist
# BUG: running eflomal doesn't yet work in biblealignlib
# go do this on internal-alignments
# -----------
#from biblealignlib.autoalign import eflomal
#efinst = eflomal.Eflomal(alset, condition)
#efinst.run_eflomal()
#efinst.run_atools()

## Regenerating Burrito Data

Details: [30AutoAligners > Regenerating Burrito Data](30AutoAligners.ipynb#Regenerating-Burrito-Data)

In [11]:
pr = reader.PharaohReader(targetlang=targetlang,
                          targetid=targetid,
                          sourceid=sourceid,
                          # must match eflomal output name: optional to set it here
                          condition=condition)
pr.make_burrito()


        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/targets/IRVBen/nt_IRVBen.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/alignments/IRVBen/SBLGNT-IRVBen-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/alignments/IRVBen/SBLGNT-IRVBen-manual.toml
        
Dropping 2 bad alignment records. Instances in self.alignmentsreader.badrecords.
Target selector is included in multiple records	2


## Instantiating a Scorer

Details: [Scoring Alignment Data > Instantiating a Scorer](31ScoringAlignmentData.ipynb#Instantiating-a-Scorer)

In [14]:
# this directory should already exist and have burrito format alignments
conditiondir = alset.langdatapath.parent / f"exp/{targetid}/{condition}"
print(f"Conditiondir: {conditiondir}")
sc = scorer.Scorer(referenceset=alset, 
                   hypothesispath=(conditiondir / f"{sourceid}-{targetid}-eflomal.json"),
                   hypothesisaltid="eflomal")

Conditiondir: /Users/sboisen/git/Clear-Bible/alignments-ben/exp/IRVBen/20250110eflomal

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/targets/IRVBen/nt_IRVBen.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/alignments/IRVBen/SBLGNT-IRVBen-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/alignments/IRVBen/SBLGNT-IRVBen-manual.toml
        
Dropping 2 bad alignment records. Instances in self.alignmentsreader.badrecords.
Target selector is included in multiple records	2
----- Hypothesis data: <AlignmentSet: ben, SBLGNT-IRVBen-eflomal>

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-ben/data/targets/IRVBen/nt_IRVBen.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-ben/exp/IRVBen/20250110

### Score the Corpus

In [15]:
print(sc.score_all().summary())

all: AER=0.4400	P=0.5600	R=0.3887	F1=0.4589


### Visualize Scores for Gospels + Acts


In [29]:
import altair as alt
import pandas as pd

# MAT-ACT
#partialscore = sc.score_partial(startbcv="40001001", endbcv="44028031")
# ROM-REV
partialscore = sc.score_partial(startbcv="45001001", endbcv="66022020")

source = pd.DataFrame([vs.asdict() for vs in partialscore.verse_scores])


In [32]:
range = "ROM-REV"
chart = alt.Chart(source, title=f"{range} F1 Scores ({sourceid}-{targetid})").mark_rect().encode(
    x='Verse:O',
    y='Chapter:O',
    color='F1:Q',
    tooltip=['Reference', 'F1', 'Precision', 'Recall']
)
chart.save(f'{range}.html')
chart

alt.Chart(...)